# Assignment 4: Random Forests for Network Intrusion Detection

## Library and Data Imports

In [7]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import utils
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.feature_selection import mutual_info_classif
from sklearn.preprocessing import OneHotEncoder, RobustScaler, FunctionTransformer
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
from sklearn.model_selection import StratifiedKFold

In [8]:
features_df = pd.read_csv('features.csv')
network_traffic_df = pd.read_csv('networkTraffic.csv', na_values='?')
attack_df = pd.read_csv('attack_category_map.csv')

In [3]:
RANDOM_STATE=10

## Data Preprocessing (aligns to that of the classification tree)

In [9]:
# drop id column
network_traffic_df = network_traffic_df.drop(columns='id')

# fix issues in features_df
features_df.columns = features_df.columns.str.strip()
features_df['Name'] = features_df['Name'].str.lower()

# split numerical and categorical columns
is_numeric = lambda col: features_df[features_df['Name'] == col]['Type'].values in ['Float', 'Integer', 'integer']
numeric = [col for col in network_traffic_df.columns if is_numeric(col) and col != 'attack_cat'] + ['rate']
categorical = [col for col in network_traffic_df.columns if col not in numeric and col != 'attack_cat' and col != 'rate']

/var/folders/pr/05vbp6_94f3300tknkj0jbw40000gn/T/ipykernel_3190/1250961649.py:9: DeprecationWarning: The truth value of an empty array is ambiguous. Returning False, but in future this will result in an error. Use `array.size > 0` to check that an array is not empty.
  is_numeric = lambda col: features_df[features_df['Name'] == col]['Type'].values in ['Float', 'Integer', 'integer']


In [10]:
wrongly_numeric = []

In [11]:
# drop service column, put service_missing indicator in its place (because it is 54.8% missing
network_traffic_df['service_missing'] = network_traffic_df['service'].isna().astype(int)
network_traffic_df = network_traffic_df.drop(columns=['service'])

## Model

In [13]:
nominal_cols = categorical + ['service_missing']
nom_idx = [network_traffic_df.columns.get_loc(col) for col in nominal_cols if col in network_traffic_df.columns]


In [14]:
from imblearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import ParameterGrid
from sklearn.metrics import f1_score
from tempfile import mkdtemp
from shutil import rmtree
import numpy as np

cache_dir = mkdtemp()
X, y = network_traffic_df.drop(columns=['attack_cat']), network_traffic_df['attack_cat']

nominal_cols = categorical + ['service_missing']
nom_idx = [X.columns.get_loc(c) for c in nominal_cols if c in X.columns]

# SMOTE-NC dropped to prevent contamination of OOB estimates
preprocessor = ColumnTransformer(
    transformers=[('onehotencode', OneHotEncoder(handle_unknown='ignore'), nom_idx)],
    remainder='passthrough'
)

param_grid = {
    'classifier__max_features': ['sqrt', 'log2', 0.3],
    'classifier__min_samples_leaf': [1, 3, 10],
    'classifier__max_depth': [None, 20],
    'classifier__class_weight': [None, 'balanced_subsample'],
}

def oob_f1_macro(rf, y):
    # out of bag f1 macro estimate
    oob_pred = rf.classes_[np.argmax(rf.oob_decision_function_, axis=1)]
    return f1_score(y, oob_pred, average='macro')

results = []
for params in ParameterGrid(param_grid):
    pipeline = Pipeline(steps=[
        ('preprocess', preprocessor),
        ('classifier', RandomForestClassifier(
            n_estimators=300,
            bootstrap=True,
            oob_score=True,
            n_jobs=-1,
            random_state=RANDOM_STATE))
    ], memory=cache_dir)
    pipeline.set_params(**params)
    pipeline.fit(X, y)

    score = oob_f1_macro(pipeline.named_steps['classifier'], y)
    results.append((params, score, pipeline))
    print(params, f"OOB macro-F1: {score:.4f}")

best_params, best_score, best_rf_model = max(results, key=lambda r: r[1])
print(best_params)
print(best_score)
rmtree(cache_dir)

{'classifier__class_weight': None, 'classifier__max_depth': None, 'classifier__max_features': 'sqrt', 'classifier__min_samples_leaf': 1} OOB macro-F1: 0.5572
{'classifier__class_weight': None, 'classifier__max_depth': None, 'classifier__max_features': 'sqrt', 'classifier__min_samples_leaf': 3} OOB macro-F1: 0.5335
{'classifier__class_weight': None, 'classifier__max_depth': None, 'classifier__max_features': 'sqrt', 'classifier__min_samples_leaf': 10} OOB macro-F1: 0.4889
{'classifier__class_weight': None, 'classifier__max_depth': None, 'classifier__max_features': 'log2', 'classifier__min_samples_leaf': 1} OOB macro-F1: 0.5485
{'classifier__class_weight': None, 'classifier__max_depth': None, 'classifier__max_features': 'log2', 'classifier__min_samples_leaf': 3} OOB macro-F1: 0.5054
{'classifier__class_weight': None, 'classifier__max_depth': None, 'classifier__max_features': 'log2', 'classifier__min_samples_leaf': 10} OOB macro-F1: 0.4524
{'classifier__class_weight': None, 'classifier__ma

In [15]:
best_params

{'classifier__class_weight': 'balanced_subsample',
 'classifier__max_depth': None,
 'classifier__max_features': 0.3,
 'classifier__min_samples_leaf': 3}

In [16]:
best_score

0.6223010665904923

In [17]:
best_rf_model

Pipeline(memory='/var/folders/pr/05vbp6_94f3300tknkj0jbw40000gn/T/tmpplxkgham',
         steps=[('preprocess',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('onehotencode',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  [1, 2, 35, 40, 41])])),
                ('classifier',
                 RandomForestClassifier(class_weight='balanced_subsample',
                                        max_features=0.3, min_samples_leaf=3,
                                        n_estimators=300, n_jobs=-1,
                                        oob_score=True, random_state=10))])